# 13 — DeepSeekMoE

**Before:** notebook **12** (MLA).

**This notebook:** sparse FFN — router + top-k experts + shared expert.

**Learning objectives**

- Run one SwiGLU expert and full `DeepSeekMoE` block.
- Trace router softmax, top-k experts, and renormalization.
- Compare dense GPT MLP params vs MoE params on one block.
- Relate router logic to `c/deepseek_v2/moe.c`.

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.

**Dojo (optional):** `dojo-grade --lesson C2-L13`


**Before:** notebook **12** (MLA).

**After:** notebook **14** (full model).

---

## GPT-2 FFN in llm.c (what you are replacing)

In `vendor/llm.c/train_gpt2.c` each block does:

1. `matmul` expand to `4*C`
2. `gelu_forward`
3. `matmul` project back to `C`

**Every token** uses the **same** MLP weights.

## DeepSeekMoE (new idea)

1. **Router** `gate(x)` → softmax over `E` experts
2. **top-k** experts per token (only run those MLPs)
3. **Weighted sum** of expert outputs
4. **Shared** expert(s) always run (stability)

**C port:** `c/deepseek_v2/moe.c` — `make test_moe`.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import c_dir, checkpoint_path, data_path, v2_checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
C_DIR = c_dir(ROOT)
V2_CKPT = v2_checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


## SwiGLU expert (not GELU)

Each expert is **SwiGLU** (Llama/DeepSeek family):

```text
hidden = silu(x @ W1) * (x @ W3)
out    = hidden @ W2
```

PyTorch: `SwiGLUExpert` in `llmc/deepseek_v2.py`.


In [ ]:
import torch
import torch.nn.functional as F
from llmc.deepseek_v2 import DeepSeekV2Config, DeepSeekMoE, SwiGLUExpert

cfg = DeepSeekV2Config.tiny(vocab_size=128, block_size=32)
x = torch.randn(2, 16, cfg.n_embd)

# One expert in isolation (like learning one MLP in GPT-2)
expert = SwiGLUExpert(cfg.n_embd, cfg.moe_intermediate)
one = expert(x[0, 0:1])
print("single expert out shape:", tuple(one.shape))

moe = DeepSeekMoE(cfg)
y = moe(x)
print("MoE batch out:", tuple(y.shape))
print("experts:", cfg.n_routed_experts, "top-k:", cfg.num_experts_per_tok, "shared:", cfg.n_shared_experts)


In [ ]:
# --- Hand-walk ONE token through the router (matches moe.c) ---
flat = x.view(-1, cfg.n_embd)
tok0 = flat[0:1]

logits = moe.gate(tok0)                    # (1, E)
probs = F.softmax(logits, dim=-1)          # router
topw, topi = torch.topk(probs, cfg.num_experts_per_tok, dim=-1)
topw = topw / topw.sum(dim=-1, keepdim=True)  # renormalize (same as C)

print("token 0 logits (first 4):", logits[0, :4].detach().round(decimals=3).tolist())
print("token 0 probs    (first 4):", probs[0, :4].detach().round(decimals=3).tolist())
print("picked expert ids:", topi[0].tolist())
print("picked weights:   ", topw[0].round(decimals=3).tolist())


In [ ]:
# --- Compare param count: one GPT MLP vs MoE stack (intuition) ---
from llmc.model import GPT, GPTConfig

gcfg = GPTConfig.tiny(128, 32)
gcfg.n_embd = cfg.n_embd
gpt = GPT(gcfg)
block = gpt.transformer.h[0]
mlp_params = sum(p.numel() for p in block.mlp.parameters())
moe_params = sum(p.numel() for p in moe.parameters())
print("GPT-2 one block MLP params:", f"{mlp_params:,}")
print("DeepSeekMoE params (this block):", f"{moe_params:,}")
print("(MoE has many experts — total compute per token is still sparse via top-k)")


## C exercise

```bash
cd c && make test_moe && ./bin/test_moe
```

Compare `token 0 routed experts` in C to `picked expert ids` above (indices should match logic; values differ if weights differ).

**Next:** notebook **14** — stack MLA + MoE + RMSNorm into one block (`make test_block`).


In [ ]:
RUN_C = False  # set True to compile/run C smokes

# Optional: run C MoE test from notebook
import shutil, subprocess
if RUN_C and shutil.which("make"):
    r = subprocess.run(["make", "test_moe"], cwd=str(C_DIR), capture_output=True, text=True)
    print(r.stdout or r.stderr)
